#***Data Pre-processing***

In [ ]:
dataset_path = "/kaggle/input/trashnet/dataset-resized"
classes = ["glass", "paper", "cardboard", "plastic", "metal", "trash"]

images = []
labels = []

# Load original images
for idx, cls in enumerate(classes):
    cls_path = os.path.join(dataset_path, cls)
    count = 0
    for img_name in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_name)
        # Resize to 224x224 and normalize pixel values to [0,1]
        img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
        img = tf.keras.utils.img_to_array(img) / 255.0
        images.append(img)
        labels.append(idx)
        count += 1
    print(f"{cls}: {count} images")

# Data augmentation pipeline (flip, rotation, brightness)
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomBrightness(0.2)
])

# Target count per class to reduce imbalance (at least 450 images)
target_counts = {"cardboard": 450, "metal": 450, "trash": 450}

# Apply augmentation only to underrepresented classes
for cls_name, target in target_counts.items():
    cls_idx = classes.index(cls_name)
    cls_images = [img for img, lbl in zip(images, labels) if lbl == cls_idx]
    current = len(cls_images)
    needed = target - current

    if needed > 0:
        aug_per_image = needed // current + 1  # How many augmented copies per image
        for img in cls_images:
            if needed <= 0:
                break
            img_batch = np.expand_dims(img, 0)
            for _ in range(aug_per_image):
                if needed <= 0:
                    break
                aug_img = augment(img_batch, training=True)[0].numpy()
                images.append(aug_img)
                labels.append(cls_idx)
                needed -= 1

# Count final distribution
from collections import Counter
final_counts = Counter(labels)
for i, cls in enumerate(classes):
    print(f"{cls}: {final_counts[i]} images (original + augmented)")

X = np.array(images)
y = tf.keras.utils.to_categorical(labels, num_classes=6)  # One-hot encode labels

# Split data: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y  # Preserve class distribution
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nTrain: {X_train.shape[0]} | Validation: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print("Done.")

glass: 501 images
paper: 594 images
cardboard: 403 images
plastic: 482 images
metal: 410 images
trash: 137 images
glass: 501 images (original + augmented)
paper: 594 images (original + augmented)
cardboard: 450 images (original + augmented)
plastic: 482 images (original + augmented)
metal: 450 images (original + augmented)
trash: 450 images (original + augmented)

Train: 2048 | Validation: 439 | Test: 440
Done.
